# MBA Friction Hunter
**Find where AI creates real learning value in an MBA curriculum — then build a working prototype.**

This notebook runs a 6-stage analysis pipeline on any MBA syllabus and outputs:
- A friction map of where students get stuck
- 2–4 prioritized AI use cases (with an approval loop so you control what moves forward)
- Architecture recommendations grounded in current research
- A design brief with realistic dialogue examples
- A working Claude system prompt ready to deploy

---

## Step 1 — Install & configure

In [ ]:
!pip install -q anthropic pdfplumber python-docx python-pptx
!git clone -q https://github.com/johnschwab1/MBA_friction_hunter.git
import os
os.chdir('MBA_friction_hunter')
print('Ready.')

In [ ]:
from google.colab import userdata
import os
os.environ['ANTHROPIC_API_KEY'] = userdata.get('Claude_A')
print('API key loaded.')

## Step 2 — Load shared utilities

In [ ]:
import re, anthropic
from pathlib import Path
from IPython.display import Markdown, display

MODEL = 'claude-opus-4-8'
client = anthropic.Anthropic()

def extract_system_prompt(md_path):
    text = Path(md_path).read_text()
    match = re.search(r'## System Prompt\s*\n(.*)', text, re.DOTALL)
    if not match:
        raise ValueError(f'No System Prompt section in {md_path}')
    return match.group(1).strip()

def call_claude(system, user, use_search=False, max_tokens=8192):
    tools = [{'type': 'web_search_20250305', 'name': 'web_search', 'max_uses': 5}] if use_search else []
    messages = [{'role': 'user', 'content': user}]
    for _ in range(20):
        kwargs = dict(model=MODEL, max_tokens=max_tokens, system=system, messages=messages)
        if tools: kwargs['tools'] = tools
        resp = client.messages.create(**kwargs)
        text_parts, tool_uses = [], []
        for block in resp.content:
            if hasattr(block, 'text'): text_parts.append(block.text)
            if getattr(block, 'type', '') == 'tool_use': tool_uses.append(block)
        if resp.stop_reason == 'end_turn' or not tool_uses:
            return '\n'.join(text_parts)
        messages.append({'role': 'assistant', 'content': resp.content})
        messages.append({'role': 'user', 'content': [
            {'type': 'tool_result', 'tool_use_id': tu.id, 'content': ''} for tu in tool_uses
        ]})
    return '\n'.join(text_parts)

outputs = {}
criteria_text = Path('stage_1_criteria_layer.md').read_text()
print('Utilities loaded.')

## Step 3 — Load the Syllabus

| Option | Formats |
|--------|---------|
| **1 — Search** | Find a public syllabus by name |
| **2 — Upload** | PDF, DOCX, PPTX, TXT |
| **3 — Google Drive** | PDF, DOCX, PPTX, TXT (export Google Docs/Slides as PDF first) |
| **4 — Paste** | Copy-paste text directly |

In [ ]:
print('─' * 50)
print('  1  Search the web')
print('  2  Upload a file  (PDF, DOCX, PPTX, TXT)')
print('  3  Google Drive')
print('  4  Paste text')
print('─' * 50)
INTAKE = input('Choose an option [1-4]: ').strip()
print()
syllabus = ''

if INTAKE == '1':
    SEARCH_QUERY = input('What syllabus are you looking for?\n  e.g. Haas MBA marketing 2025\n> ').strip()
    print(f'\nSearching for: {SEARCH_QUERY}...')
    find_system = (
        'You are a research assistant. Search for MBA course syllabi matching the query. '
        'Return a numbered list of up to 5 options: number, title, school, year, one-sentence description. '
        'Format: N. Title — School (Year): Description'
    )
    search_results = call_claude(find_system, f'Find MBA syllabi for: {SEARCH_QUERY}', use_search=True, max_tokens=1000)
    print()
    print(search_results)
    print()
    PICK = input('Which result? Enter a number: ').strip()
    print('\nFetching full syllabus...')
    fetch_system = (
        'You are a research assistant. From the search results, fetch the full text of the selected option. '
        'Return only the complete syllabus: description, objectives, schedule, assignments, readings.'
    )
    syllabus = call_claude(fetch_system,
        f'Results:\n\n{search_results}\n\nFetch full text of option {PICK}.',
        use_search=True, max_tokens=4096)
    print(f'✓ Fetched {len(syllabus):,} chars.')

elif INTAKE == '2':
    from google.colab import files as colab_files
    import io
    print('Select your file (PDF, DOCX, PPTX, or TXT):')
    print('For Google Docs/Slides: export as PDF first (File > Download > PDF).')
    uploaded = colab_files.upload()
    if not uploaded:
        print('⚠ No file selected.')
    else:
        filename = list(uploaded.keys())[0]
        ext = filename.rsplit('.', 1)[-1].lower()
        raw = uploaded[filename]
        if ext == 'pdf':
            import pdfplumber, io
            with pdfplumber.open(io.BytesIO(raw)) as pdf:
                syllabus = '\n\n'.join(p.extract_text() or '' for p in pdf.pages)
        elif ext == 'docx':
            import docx, io
            doc = docx.Document(io.BytesIO(raw))
            syllabus = '\n'.join(p.text for p in doc.paragraphs)
        elif ext == 'pptx':
            from pptx import Presentation
            import io
            prs = Presentation(io.BytesIO(raw))
            slides = []
            for i, slide in enumerate(prs.slides, 1):
                texts = [s.text.strip() for s in slide.shapes if hasattr(s, 'text') and s.text.strip()]
                if texts:
                    slides.append(f'[Slide {i}]\n' + '\n'.join(texts))
            syllabus = '\n\n'.join(slides)
        else:
            syllabus = raw.decode('utf-8', errors='replace')
        print(f'✓ Loaded {filename!r} — {len(syllabus):,} chars.')

elif INTAKE == '3':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print()
    print('Note: Google Docs and Slides cannot be read directly.')
    print('Export first: File > Download > PDF or DOCX, then use that file path here.')
    print()
    DRIVE_PATH = input('File path in your Drive:\n  e.g. /content/drive/MyDrive/syllabi/marketing.pdf\n> ').strip()
    ext = DRIVE_PATH.rsplit('.', 1)[-1].lower()
    if ext == 'pdf':
        import pdfplumber
        with pdfplumber.open(DRIVE_PATH) as pdf:
            syllabus = '\n\n'.join(p.extract_text() or '' for p in pdf.pages)
    elif ext == 'docx':
        import docx
        doc = docx.Document(DRIVE_PATH)
        syllabus = '\n'.join(p.text for p in doc.paragraphs)
    elif ext == 'pptx':
        from pptx import Presentation
        prs = Presentation(DRIVE_PATH)
        slides = []
        for i, slide in enumerate(prs.slides, 1):
            texts = [s.text.strip() for s in slide.shapes if hasattr(s, 'text') and s.text.strip()]
            if texts:
                slides.append(f'[Slide {i}]\n' + '\n'.join(texts))
        syllabus = '\n\n'.join(slides)
    else:
        with open(DRIVE_PATH, 'r', errors='replace') as f:
            syllabus = f.read()
    print(f'✓ Loaded from Drive — {len(syllabus):,} chars.')

elif INTAKE == '4':
    print('Paste your syllabus. Type END on its own line when done:')
    lines = []
    while True:
        line = input()
        if line.strip().upper() == 'END': break
        lines.append(line)
    syllabus = '\n'.join(lines)
    print(f'✓ Received {len(syllabus):,} chars.')

if syllabus.strip():
    print()
    print('PREVIEW (first 500 chars):')
    print('─' * 50)
    print(syllabus[:500] + ('…' if len(syllabus) > 500 else ''))
else:
    print('⚠ No syllabus loaded. Re-run this cell.')

---
## Stage 2 — Friction Hunter
*Classifies where students get stuck and scores each friction type.*

In [ ]:
print('Running Stage 2: Friction Hunter...')
system2 = extract_system_prompt('stage_2_friction_hunter.md')
user2 = f'CRITERIA REFERENCE:\n\n{criteria_text}\n\n---\n\nCURRICULUM TO ANALYZE:\n\n{syllabus}'
outputs[2] = call_claude(system2, user2)
print('Done.')
display(Markdown(outputs[2]))

---
## Stage 3 — Use Case Selector
*Picks 2–4 use cases. Runs in a loop — review the output, give direction, and re-run until satisfied.*

In [ ]:
print('─' * 50)
print('Stage 3: Use Case Selector')
print('─' * 50)
print()

system3 = extract_system_prompt('stage_3_use_case_selector.md')
stage3_direction = ''

while True:
    if stage3_direction:
        user3 = outputs[2] + '\n\nDIRECTION FROM PROFESSOR:\n' + stage3_direction
    else:
        user3 = outputs[2]

    note = ' (with your direction)' if stage3_direction else ''
    print(f'Analyzing use cases{note}...')
    result = call_claude(system3, user3)
    display(Markdown(result))

    print('Generating guidance suggestions...')
    ex_system = (
        'Based on these MBA course AI use case recommendations, write 3-4 short, specific '
        'examples of direction a professor might give to improve or redirect the selection. '
        'Make each concrete and tied to the actual use cases shown. '
        'Reference specific course elements, weeks, or assignments by name. '
        'Format as a markdown bulleted list. Each should be one complete actionable sentence.'
    )
    examples = call_claude(ex_system, 'Use cases:\n\n' + result, max_tokens=400)

    print()
    print('─' * 50)
    print('Not sure what to change? Here are examples of useful direction:')
    display(Markdown(examples))
    print('─' * 50)
    print()

    print('Are you satisfied with these use cases?')
    print('  1  Yes — continue to Stage 4')
    print('  2  Re-run with new direction')
    choice = input('> ').strip()

    if choice == '1':
        outputs[3] = result
        print()
        print('✓ Use cases confirmed. Ready for Stage 4.')
        break

    print()
    print('What should I change or focus on?')
    print('  e.g. "The case competition in weeks 6-8 is the core of this course"')
    stage3_direction = input('> ').strip()
    print()
    print('Re-running with your direction...')
    print()

---
## Stage 4 — Architecture Hunter
> Web search is active for this stage.

In [ ]:
print('Running Stage 4: Architecture Hunter (web search active)...')
system4 = extract_system_prompt('stage_4_architecture_hunter.md')
outputs[4] = call_claude(system4, outputs[3], use_search=True)
print('Done.')
display(Markdown(outputs[4]))

---
## Stage 5 — Design Brief

In [ ]:
print('Running Stage 5: Design Brief...')
system5 = extract_system_prompt('stage_5_design_brief_generator.md')
user5 = (
    f'PRIORITIZED USE CASES (Stage 3):\n\n{outputs[3]}\n\n---\n\n'
    f'ARCHITECTURE RECOMMENDATIONS (Stage 4):\n\n{outputs[4]}'
)
outputs[5] = call_claude(system5, user5)
print('Done.')
display(Markdown(outputs[5]))

---
## Stage 6 — Prototype

In [ ]:
print('Running Stage 6: Prototype Generator...')
system6 = extract_system_prompt('stage_6_prototype_generator.md')
outputs[6] = call_claude(system6, outputs[5])
print('Done.')
display(Markdown(outputs[6]))

---
## Save all outputs

In [ ]:
from pathlib import Path
from datetime import datetime
stage_names = {2: 'friction_hunter', 3: 'use_case_selector', 4: 'architecture_hunter', 5: 'design_brief', 6: 'prototype'}
run_name = datetime.now().strftime('%Y%m%d_%H%M%S')
out_dir = Path(f'output/{run_name}')
out_dir.mkdir(parents=True, exist_ok=True)
for stage_num, content in outputs.items():
    path = out_dir / f'stage_{stage_num:02d}_{stage_names[stage_num]}.md'
    path.write_text(content)
    print(f'Saved {path}')
print(f'\nAll outputs saved to {out_dir}')